# 05 - Benchmark, Analysis, and Paper Outputs

**Design:** 30 queries x 3 topologies x 3 repeats x 2 arms = **540 runs**.

- Topologies: `all` (sequential), `parallel` (concurrent), `dag` (converging).
- Arms: `peft` (adapters loaded) and `base` (identical models, no adapters).

The `base` arm is the control that makes RQ2 answerable. Without it, a topology
ranking over specialised agents cannot be distinguished from one that would have
come out the same way with no specialisation at all.

`single`, `legal_first`, `planner_based`, `verify_only` and
`legal_news_parallel` remain implemented and selectable, but are out of scope.

## 5.1 Preflight

Run these before committing to a multi-hour job. Each guards a failure that
otherwise produces a complete-looking but invalid dataset.

In [ ]:
# 1. Corpus present (several hundred chunks, not tens)
import utils; print('chunks:', utils.get_db_document_count())

In [ ]:
# 2. Ollama up - embeddings only, but retrieval silently degrades without it
import config, requests
print(requests.get(config.OLLAMA_BASE_URL + '/api/tags', timeout=8).status_code)

In [ ]:
# 3. Code invariants. Must all pass - these guard the defects that
#    invalidated earlier runs.
!.venv\Scripts\python.exe -m pytest tests -q

In [ ]:
# 4. Adapters present and non-inert
!.venv\Scripts\python.exe finetune\validate_adapters.py

In [ ]:
# 5. Smoke test - 1 query x 3 topologies x 1 repeat PER ARM, separate output file
!powershell -File run_experiment.ps1 -Smoke

### Inspect the smoke answers - the most important check in the process

Any of these means **stop**, not proceed:

| observation | cause |
|---|---|
| abstention sentence in every mode | retrieval is empty |
| `peft` and `base` answers identical | adapters inert, or the arm switch is not taking effect |
| all topologies identical within an arm | sampling off; `LEGALAI_DETERMINISTIC` must be `0` |
| canned GPAI provider-tier template | COMPL-AI enabled, bypassing the graph |

In [ ]:
import json
for r in map(json.loads, open('benchmark_runs_smoke.jsonl', encoding='utf-8')):
    if r.get('success'):
        print(r['arm'].ljust(5), r['mode'].ljust(9), f"{r['elapsed_s']:7.1f}s",
              '|', repr(r.get('response', ''))[:110])

## 5.2 The full run

`-BenchmarkOnly` stops before analysis, because the reference answers still need
human review. Runs both arms in sequence (`peft` then `base`), appending into one
`benchmark_runs.jsonl` distinguished by an `arm` column.

In [ ]:
!powershell -File run_experiment.ps1 -BenchmarkOnly

# If it dies partway, resume without losing work:
#   powershell -File run_experiment.ps1 -Resume -BenchmarkOnly
# Rows are flushed after every single run, so a crash at run 400 keeps 399.

### Operational notes

- **Do not run anything else GPU-heavy.** With ~180 MiB spare VRAM another CUDA
  process causes an out-of-memory failure, not just noisy latency.
- **Mains power, sleep disabled.** Battery throttling distorts the latency the
  paper reports; sustained load already took step times from 5.2 to 9.7 s during
  training.
- `REQUEST_TIMEOUT_S` is **3600**, not 600. The validator can send the whole graph
  round again (`MAX_ITERATIONS=2`), and retries are the common case - 508 of 630
  pre-pivot rows show 20+ graph steps against 14-17 for a single pass. A
  10-minute ceiling would convert ordinary behaviour into recorded failures and
  bias results toward whichever topology retries least.

In [ ]:
# Abstention rate per arm and topology - report this.
import json, collections
rows = [json.loads(l) for l in open('benchmark_runs.jsonl', encoding='utf-8') if l.strip()]
c = collections.Counter((r['arm'], r['mode'], bool(r.get('abstained')))
                        for r in rows if r.get('success'))
for (arm, mode, ab), n in sorted(c.items()):
    print(arm.ljust(5), mode.ljust(9), 'abstained' if ab else 'answered', n)

## 5.3 Human steps - only Charbel can do these

1. **Review the 30 reference answers** in `eval_dataset.json`. They were drafted
   by a model and are all flagged `needs_review: true`. Every quality metric is
   measured against them.
2. **Relevance annotation** (`scripts/annotate_relevance.py`), or accept that
   retrieval metrics stay suppressed. Do not re-enable them any other way: the
   previous labels were copies of the retriever's own output, which made
   precision@5 equal to 1.0 by construction.

## 5.4 Analysis

Two comparison families, Holm-corrected **within each family separately**:

- **RQ1/RQ3** - all three pairs of topologies, run *within* an arm (never across,
  which would confound structure with specialisation).
- **RQ2** - `peft` vs `base` within each topology.

Experimental unit is the **query**; repeats are averaged within each
(query, topology, arm) cell first. A positive median difference and positive
Cliff's delta both mean the **first-named** side scored higher.

In [ ]:
# Judge sanity + cost before spending on 540 answers
!.venv\Scripts\python.exe llm_judge.py --check

In [ ]:
!.venv\Scripts\python.exe analyze_results.py

# Watch for:
#   'WARNING some rows have no arm field'          -> pre-pivot rows mixed in
#   'NOTE no ablation results'                     -> RQ2 unanswerable, drop it
#   'WARNING latency drifted N% within an arm'     -> between-arm latency
#                                                     comparison uninterpretable
#   'WARNING gold answers still flagged needs_review'

### The latency-drift check

The arms run as **consecutive passes**, not interleaved (six co-resident models
will not fit, so adapters cannot be swapped per query). Anything varying slowly
over a multi-day run - thermal throttling above all - therefore falls
disproportionately on whichever arm ran second, and `elapsed_s` is one of the
tested metrics.

Every run records `started_at_utc`; the analysis reports within-arm drift between
the first and last decile and warns above 15%. It **reports** rather than
corrects: regressing latency on wall-clock time would hide a measurement problem
behind a statistical one. Topology comparisons are unaffected - modes interleave
within an arm.

In [ ]:
!.venv\Scripts\python.exe evaluate_workflows.py      # metrics_table.tex + ablation table
!.venv\Scripts\python.exe make_paper_figures.py      # fig1-fig5
!.venv\Scripts\python.exe make_topology_figure.py    # fig0

In [ ]:
!.venv\Scripts\python.exe scripts\print_summary.py

## 5.5 Paper outputs

Upload to Overleaf: `main.tex`, `references.bib`, `metrics_table.tex`,
`metrics_table_ablation.tex`, and `paper_figures/*.png` into `figures/`.

`main.tex` uses `\TBD{}` for every unfilled quantitative claim, rendered in red.
**A `\TBD{}` surviving into a submitted PDF is a bug, not a rounding decision.**
They are filled from `results.json` - never typed in by hand.

In [ ]:
# Count remaining placeholders
import re
tex = open('../main.tex', encoding='utf-8').read()
print('TBD placeholders:', tex.count('\\TBD{'))
labels = set(re.findall(r'\\label\{([^}]+)\}', tex))
refs = set(re.findall(r'\\ref\{([^}]+)\}', tex))
print('dangling refs:', sorted(refs - labels) or 'none')

In [ ]:
# Snapshot the run. Provenance is a hash of the source files that produced the
# numbers - this project is kept fully local with no git repository, so there is
# no commit id. Arguably better: it reflects the code that actually ran.
!.venv\Scripts\python.exe scripts\snapshot_run.py --label peft-pivot